# March ML Mania 2026 - Full Training Pipeline
**Three-Tier Ensemble: GBM + Deep Learning + Foundation Models**

Pipeline: Feature Engineering → Elo → Massey Ordinals → Traditional ML → DL → Ensemble → Submission

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Kaggle vs local
import os
IS_KAGGLE = os.path.exists("/kaggle/input")
if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/competitions/march-machine-learning-mania-2026")
    OUT_DIR = Path("/kaggle/working")
    # !pip install -q optuna shap
else:
    DATA_DIR = Path(__file__).parent.parent / "data" / "raw"
    OUT_DIR = Path(__file__).parent.parent / ".tmp"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
PRED_CLIP_MIN, PRED_CLIP_MAX = 0.05, 0.95
CURRENT_SEASON = 2026

## 1. Data Loading

In [ ]:
print("Loading data...")
m_teams = pd.read_csv(DATA_DIR / "MTeams.csv")
m_reg_compact = pd.read_csv(DATA_DIR / "MRegularSeasonCompactResults.csv")
m_reg_detailed = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
m_tourney_compact = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")
m_seeds = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
m_conferences = pd.read_csv(DATA_DIR / "MTeamConferences.csv")
m_coaches = pd.read_csv(DATA_DIR / "MTeamCoaches.csv")
m_massey = pd.read_csv(DATA_DIR / "MMasseyOrdinals.csv")
m_conf_tourney = pd.read_csv(DATA_DIR / "MConferenceTourneyGames.csv")

w_reg_compact = pd.read_csv(DATA_DIR / "WRegularSeasonCompactResults.csv")
w_reg_detailed = pd.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")
w_tourney_compact = pd.read_csv(DATA_DIR / "WNCAATourneyCompactResults.csv")
w_seeds = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")
w_conferences = pd.read_csv(DATA_DIR / "WTeamConferences.csv")
w_coaches = pd.read_csv(DATA_DIR / "MTeamCoaches.csv")  # Men's coaches for now
w_massey = pd.read_csv(DATA_DIR / "WGameCities.csv")  # placeholder

sub1 = pd.read_csv(DATA_DIR / "SampleSubmissionStage1.csv")
sub2 = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")

# Parse seeds
m_seeds['SeedNum'] = m_seeds['Seed'].str[1:3].astype(int)
w_seeds['SeedNum'] = w_seeds['Seed'].str[1:3].astype(int)

print(f"Men's: {len(m_reg_compact)} reg games, {len(m_tourney_compact)} tourney games")
print(f"Women's: {len(w_reg_compact)} reg games, {len(w_tourney_compact)} tourney games")
print(f"Massey ordinals: {len(m_massey):,} entries")

## 2. Feature Engineering - Phase 3

### 2.1 Elo Rating System

In [ ]:
class EloSystem:
    def __init__(self, k=32, home_adv=100, margin_mult=0.006, reversion=0.25):
        self.k = k
        self.home_adv = home_adv
        self.margin_mult = margin_mult
        self.reversion = reversion
        self.ratings = {}
        self.initial = 1500

    def get(self, team):
        return self.ratings.get(team, self.initial)

    def expected(self, ra, rb):
        return 1.0 / (1.0 + 10.0 ** ((rb - ra) / 400.0))

    def update(self, winner, loser, margin, wloc='N'):
        rw, rl = self.get(winner), self.get(loser)
        # Home court
        if wloc == 'H':
            rw_adj, rl_adj = rw + self.home_adv, rl
        elif wloc == 'A':
            rw_adj, rl_adj = rw, rl + self.home_adv
        else:
            rw_adj, rl_adj = rw, rl

        exp_w = self.expected(rw_adj, rl_adj)
        # Margin-of-victory multiplier
        mov = np.log(abs(margin) + 1) * (2.2 / (abs(rw - rl) * self.margin_mult + 2.2))
        adj = self.k * mov * (1 - exp_w)
        self.ratings[winner] = rw + adj
        self.ratings[loser] = rl - adj

    def new_season(self):
        for t in self.ratings:
            self.ratings[t] = self.ratings[t] * (1 - self.reversion) + self.initial * self.reversion

    def predict(self, a, b):
        return self.expected(self.get(a), self.get(b))

def build_elo(reg_df, tourney_df=None, k=32):
    """Build Elo ratings from game history. Returns {season: {team: rating}}."""
    elo = EloSystem(k=k)
    all_games = reg_df.copy()
    if tourney_df is not None:
        # Include past tourney games for Elo update (NOT for feature leakage)
        all_games = pd.concat([all_games, tourney_df], ignore_index=True)
    all_games = all_games.sort_values(['Season', 'DayNum']).reset_index(drop=True)

    season_ratings = {}
    prev_season = None

    for _, g in all_games.iterrows():
        if g['Season'] != prev_season:
            if prev_season is not None:
                elo.new_season()
            prev_season = g['Season']
            # Snapshot before tournament starts (DayNum < 134)

        margin = g['WScore'] - g['LScore']
        wloc = g.get('WLoc', 'N')
        elo.update(g['WTeamID'], g['LTeamID'], margin, wloc)

        # Snapshot at end of regular season (DayNum ~133)
        if 132 <= g['DayNum'] <= 133:
            if g['Season'] not in season_ratings:
                season_ratings[g['Season']] = {}
            season_ratings[g['Season']] = dict(elo.ratings)

    # Also snapshot current season at latest available day
    season_ratings[all_games['Season'].max()] = dict(elo.ratings)
    return season_ratings, elo

print("Building Men's Elo ratings...")
m_elo_ratings, m_elo = build_elo(m_reg_compact, m_tourney_compact, k=32)
print(f"  Elo ratings for {len(m_elo_ratings)} seasons")

print("Building Women's Elo ratings...")
w_elo_ratings, w_elo = build_elo(w_reg_compact, w_tourney_compact, k=32)
print(f"  Elo ratings for {len(w_elo_ratings)} seasons")

# Convert to DataFrame
def elo_to_df(season_ratings):
    rows = []
    for season, ratings in season_ratings.items():
        for team, rating in ratings.items():
            rows.append({'Season': season, 'TeamID': team, 'EloRating': rating})
    return pd.DataFrame(rows)

m_elo_df = elo_to_df(m_elo_ratings)
w_elo_df = elo_to_df(w_elo_ratings)
print(f"  Men's Elo entries: {len(m_elo_df)}, Women's: {len(w_elo_df)}")

### 2.2 Team Season Statistics (Four Factors + Efficiency)

In [ ]:
def compute_team_stats(detailed_df, compact_df):
    """Compute per-team per-season stats from detailed + compact results."""

    # From detailed data: Four Factors + efficiency
    def extract_stats(df, prefix):
        p = prefix
        o = 'L' if p == 'W' else 'W'
        r = pd.DataFrame()
        r['Season'] = df['Season']
        r['TeamID'] = df[f'{p}TeamID']
        r['DayNum'] = df['DayNum']
        r['Win'] = 1 if p == 'W' else 0
        r['Score'] = df[f'{p}Score']
        r['OppScore'] = df[f'{o}Score']
        r['FGM'] = df[f'{p}FGM']; r['FGA'] = df[f'{p}FGA']
        r['FGM3'] = df[f'{p}FGM3']; r['FGA3'] = df[f'{p}FGA3']
        r['FTM'] = df[f'{p}FTM']; r['FTA'] = df[f'{p}FTA']
        r['OR'] = df[f'{p}OR']; r['DR'] = df[f'{p}DR']
        r['Ast'] = df[f'{p}Ast']; r['TO'] = df[f'{p}TO']
        r['Stl'] = df[f'{p}Stl']; r['Blk'] = df[f'{p}Blk']
        r['OppOR'] = df[f'{o}OR']; r['OppDR'] = df[f'{o}DR']
        r['OppFGA'] = df[f'{o}FGA']; r['OppFTA'] = df[f'{o}FTA']
        r['OppTO'] = df[f'{o}TO']; r['OppFGM'] = df[f'{o}FGM']
        r['OppFGM3'] = df[f'{o}FGM3']
        return r

    all_games = pd.concat([extract_stats(detailed_df, 'W'),
                            extract_stats(detailed_df, 'L')], ignore_index=True)

    # Regular season only (DayNum < 132)
    reg_games = all_games[all_games['DayNum'] < 132]

    # Aggregate per team per season
    agg = reg_games.groupby(['Season', 'TeamID']).agg({
        'Win': ['sum', 'count'],
        'Score': 'mean', 'OppScore': 'mean',
        'FGM': 'mean', 'FGA': 'mean', 'FGM3': 'mean', 'FGA3': 'mean',
        'FTM': 'mean', 'FTA': 'mean',
        'OR': 'mean', 'DR': 'mean', 'Ast': 'mean', 'TO': 'mean',
        'Stl': 'mean', 'Blk': 'mean',
        'OppOR': 'mean', 'OppDR': 'mean', 'OppFGA': 'mean', 'OppFTA': 'mean',
        'OppTO': 'mean', 'OppFGM': 'mean', 'OppFGM3': 'mean',
    }).reset_index()
    agg.columns = ['Season', 'TeamID', 'Wins', 'Games',
                    'Score', 'OppScore', 'FGM', 'FGA', 'FGM3', 'FGA3',
                    'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk',
                    'OppOR', 'OppDR', 'OppFGA', 'OppFTA', 'OppTO', 'OppFGM', 'OppFGM3']

    # Computed features
    agg['WinPct'] = agg['Wins'] / agg['Games']
    agg['PointDiff'] = agg['Score'] - agg['OppScore']

    # Four Factors
    agg['eFG_pct'] = (agg['FGM'] + 0.5 * agg['FGM3']) / agg['FGA']
    poss = agg['FGA'] + 0.44 * agg['FTA'] + agg['TO']
    agg['TO_pct'] = agg['TO'] / poss
    agg['ORB_pct'] = agg['OR'] / (agg['OR'] + agg['OppDR'])
    agg['FT_rate'] = agg['FTM'] / agg['FGA']

    # Opponent Four Factors
    agg['Opp_eFG_pct'] = (agg['OppFGM'] + 0.5 * agg['OppFGM3']) / agg['OppFGA']
    opp_poss = agg['OppFGA'] + 0.44 * agg['OppFTA'] + agg['OppTO']
    agg['Opp_TO_pct'] = agg['OppTO'] / opp_poss

    # Efficiency
    agg['OffRating'] = agg['Score'] / poss * 100
    agg['DefRating'] = agg['OppScore'] / opp_poss * 100
    agg['NetRating'] = agg['OffRating'] - agg['DefRating']
    agg['Pace'] = (poss + opp_poss) / 2

    # Additional
    agg['FG_pct'] = agg['FGM'] / agg['FGA']
    agg['FG3_pct'] = agg['FGM3'] / agg['FGA3']
    agg['FT_pct'] = agg['FTM'] / agg['FTA']
    agg['Ast_TO_ratio'] = agg['Ast'] / agg['TO']

    # Last 10 games performance (momentum)
    last10 = reg_games.sort_values('DayNum').groupby(['Season', 'TeamID']).tail(10)
    last10_agg = last10.groupby(['Season', 'TeamID']).agg({
        'Win': 'mean', 'Score': 'mean', 'OppScore': 'mean'
    }).reset_index()
    last10_agg.columns = ['Season', 'TeamID', 'Last10_WinPct', 'Last10_Score', 'Last10_OppScore']
    last10_agg['Last10_PointDiff'] = last10_agg['Last10_Score'] - last10_agg['Last10_OppScore']

    agg = agg.merge(last10_agg, on=['Season', 'TeamID'], how='left')

    # Consistency (std dev of point differential per game)
    game_margins = reg_games.copy()
    game_margins['Margin'] = game_margins['Score'] - game_margins['OppScore']
    consistency = game_margins.groupby(['Season', 'TeamID'])['Margin'].std().reset_index()
    consistency.columns = ['Season', 'TeamID', 'MarginStd']
    agg = agg.merge(consistency, on=['Season', 'TeamID'], how='left')

    # Road performance
    # Need WLoc from compact data
    road_w = compact_df[compact_df['WLoc'] == 'A'][['Season', 'WTeamID']].copy()
    road_w.columns = ['Season', 'TeamID']
    road_w['RoadWin'] = 1
    road_l = compact_df[compact_df['WLoc'] == 'H'][['Season', 'LTeamID']].copy()
    road_l.columns = ['Season', 'TeamID']
    road_l['RoadWin'] = 0
    road_all = pd.concat([road_w, road_l])
    road_agg = road_all.groupby(['Season', 'TeamID'])['RoadWin'].agg(['mean', 'count']).reset_index()
    road_agg.columns = ['Season', 'TeamID', 'RoadWinPct', 'RoadGames']
    agg = agg.merge(road_agg, on=['Season', 'TeamID'], how='left')

    return agg

print("Computing Men's team stats...")
m_team_stats = compute_team_stats(m_reg_detailed, m_reg_compact)
print(f"  {len(m_team_stats)} team-season records, {len(m_team_stats.columns)} features")

print("Computing Women's team stats...")
w_team_stats = compute_team_stats(w_reg_detailed, w_reg_compact)
print(f"  {len(w_team_stats)} team-season records")

### 2.3 Massey Ordinals (External Rankings)

In [ ]:
# Get end-of-regular-season rankings from top systems
TOP_SYSTEMS = ['POM', 'SAG', 'MOR', 'DOL', 'COL', 'RPI', 'AP', 'USA', 'WOL', 'RTH']

def extract_massey(massey_df, systems=TOP_SYSTEMS):
    """Extract end-of-regular-season rankings from top Massey systems."""
    # Get rankings near end of regular season (DayNum 128-133)
    eos = massey_df[
        (massey_df['RankingDayNum'] >= 128) &
        (massey_df['RankingDayNum'] <= 133) &
        (massey_df['SystemName'].isin(systems))
    ].copy()

    # Take latest available ranking per season/system/team
    eos = eos.sort_values('RankingDayNum').groupby(['Season', 'SystemName', 'TeamID']).tail(1)

    # Pivot: one column per system
    pivoted = eos.pivot_table(index=['Season', 'TeamID'], columns='SystemName',
                               values='OrdinalRank', aggfunc='first').reset_index()

    # Consensus rank = mean of all available systems
    rank_cols = [c for c in pivoted.columns if c in systems]
    pivoted['ConsensusRank'] = pivoted[rank_cols].mean(axis=1)

    return pivoted

print("Extracting Massey ordinals...")
m_massey_features = extract_massey(m_massey)
print(f"  Men's Massey features: {m_massey_features.shape}")
print(f"  Available systems: {[c for c in m_massey_features.columns if c in TOP_SYSTEMS]}")

### 2.4 Coach Tournament Experience

In [ ]:
def compute_coach_features(coaches_df, tourney_df):
    """Compute coach tournament experience features."""
    # Get coach for each team at tournament time (DayNum > 132)
    tourney_coaches = coaches_df[coaches_df['LastDayNum'] >= 132].copy()
    tourney_coaches = tourney_coaches.drop_duplicates(['Season', 'TeamID'], keep='last')

    # Count prior tournament appearances per coach
    coach_tourney_wins = tourney_df.merge(
        tourney_coaches[['Season', 'TeamID', 'CoachName']],
        left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'], how='left'
    )

    # Historical tournament wins by coach (cumulative up to current season)
    coach_exp = []
    for season in sorted(tourney_coaches['Season'].unique()):
        prior_wins = coach_tourney_wins[coach_tourney_wins['Season'] < season]
        coach_wins = prior_wins.groupby('CoachName').size().to_dict()

        season_coaches = tourney_coaches[tourney_coaches['Season'] == season]
        for _, row in season_coaches.iterrows():
            coach_exp.append({
                'Season': season,
                'TeamID': row['TeamID'],
                'CoachName': row['CoachName'],
                'CoachTourneyWins': coach_wins.get(row['CoachName'], 0),
            })

    return pd.DataFrame(coach_exp)

print("Computing coach features...")
m_coach_features = compute_coach_features(m_coaches, m_tourney_compact)
print(f"  Coach features: {len(m_coach_features)} entries")

### 2.5 Conference Tournament Results

In [ ]:
def conf_tourney_features(conf_tourney_df, conferences_df):
    """Did team win their conference tournament?"""
    # The last game winner in each conference is the champion
    champs = conf_tourney_df.sort_values('DayNum').groupby(['Season', 'ConfAbbrev']).tail(1)
    champs = champs[['Season', 'WTeamID']].rename(columns={'WTeamID': 'TeamID'})
    champs['ConfTourneyChamp'] = 1

    # Games played in conf tourney
    w_games = conf_tourney_df[['Season', 'WTeamID']].rename(columns={'WTeamID': 'TeamID'})
    l_games = conf_tourney_df[['Season', 'LTeamID']].rename(columns={'LTeamID': 'TeamID'})
    all_ct = pd.concat([w_games, l_games])
    ct_games = all_ct.groupby(['Season', 'TeamID']).size().reset_index(name='ConfTourneyGames')

    # Wins
    ct_wins = conf_tourney_df.groupby(['Season', 'WTeamID']).size().reset_index(name='ConfTourneyWins')
    ct_wins.columns = ['Season', 'TeamID', 'ConfTourneyWins']

    result = ct_games.merge(ct_wins, on=['Season', 'TeamID'], how='left').fillna(0)
    result = result.merge(champs, on=['Season', 'TeamID'], how='left').fillna(0)

    return result

print("Computing conference tournament features...")
m_conf_features = conf_tourney_features(m_conf_tourney, m_conferences)
print(f"  Conf tourney features: {len(m_conf_features)} entries")

### 2.6 Build Matchup Feature Matrix

In [ ]:
def build_matchup_features(tourney_df, seeds_df, team_stats, elo_df, massey_df,
                            coach_df, conf_df, gender='M'):
    """Build training data: one row per tournament game with difference features."""
    rows = []

    for _, game in tourney_df.iterrows():
        season = game['Season']
        w_id, l_id = game['WTeamID'], game['LTeamID']

        # Ensure lower ID is team_a (submission format)
        team_a, team_b = min(w_id, l_id), max(w_id, l_id)
        target = 1 if team_a == w_id else 0  # Did lower ID win?

        features = {'Season': season, 'TeamA': team_a, 'TeamB': team_b, 'Target': target}

        # --- Seed features ---
        a_seed = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_a)]
        b_seed = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_b)]
        if len(a_seed) > 0 and len(b_seed) > 0:
            features['SeedA'] = a_seed.iloc[0]['SeedNum']
            features['SeedB'] = b_seed.iloc[0]['SeedNum']
            features['SeedDiff'] = features['SeedA'] - features['SeedB']
        else:
            continue  # Skip if no seed info

        # --- Elo features ---
        a_elo = elo_df[(elo_df['Season'] == season) & (elo_df['TeamID'] == team_a)]
        b_elo = elo_df[(elo_df['Season'] == season) & (elo_df['TeamID'] == team_b)]
        if len(a_elo) > 0 and len(b_elo) > 0:
            features['EloA'] = a_elo.iloc[0]['EloRating']
            features['EloB'] = b_elo.iloc[0]['EloRating']
            features['EloDiff'] = features['EloA'] - features['EloB']
        else:
            features['EloDiff'] = 0

        # --- Team stats difference features ---
        a_stats = team_stats[(team_stats['Season'] == season) & (team_stats['TeamID'] == team_a)]
        b_stats = team_stats[(team_stats['Season'] == season) & (team_stats['TeamID'] == team_b)]

        stat_cols = ['WinPct', 'PointDiff', 'eFG_pct', 'TO_pct', 'ORB_pct', 'FT_rate',
                     'OffRating', 'DefRating', 'NetRating', 'Pace', 'FG_pct', 'FG3_pct',
                     'FT_pct', 'Ast_TO_ratio', 'Opp_eFG_pct', 'Opp_TO_pct',
                     'Last10_WinPct', 'Last10_PointDiff', 'MarginStd', 'RoadWinPct']

        if len(a_stats) > 0 and len(b_stats) > 0:
            a, b = a_stats.iloc[0], b_stats.iloc[0]
            for col in stat_cols:
                if col in a.index and col in b.index:
                    features[f'{col}_diff'] = a[col] - b[col]
        else:
            continue

        # --- Massey ordinal features ---
        if massey_df is not None:
            a_massey = massey_df[(massey_df['Season'] == season) & (massey_df['TeamID'] == team_a)]
            b_massey = massey_df[(massey_df['Season'] == season) & (massey_df['TeamID'] == team_b)]
            if len(a_massey) > 0 and len(b_massey) > 0:
                am, bm = a_massey.iloc[0], b_massey.iloc[0]
                for sys in ['POM', 'SAG', 'MOR', 'ConsensusRank']:
                    if sys in am.index and sys in bm.index:
                        val_a = am[sys] if not pd.isna(am[sys]) else 150
                        val_b = bm[sys] if not pd.isna(bm[sys]) else 150
                        features[f'{sys}_diff'] = val_a - val_b  # Lower rank = better

        # --- Coach features ---
        if coach_df is not None:
            a_coach = coach_df[(coach_df['Season'] == season) & (coach_df['TeamID'] == team_a)]
            b_coach = coach_df[(coach_df['Season'] == season) & (coach_df['TeamID'] == team_b)]
            if len(a_coach) > 0 and len(b_coach) > 0:
                features['CoachExp_diff'] = a_coach.iloc[0]['CoachTourneyWins'] - b_coach.iloc[0]['CoachTourneyWins']

        # --- Conference tourney features ---
        if conf_df is not None:
            a_conf = conf_df[(conf_df['Season'] == season) & (conf_df['TeamID'] == team_a)]
            b_conf = conf_df[(conf_df['Season'] == season) & (conf_df['TeamID'] == team_b)]
            if len(a_conf) > 0 and len(b_conf) > 0:
                features['ConfTourneyWins_diff'] = a_conf.iloc[0].get('ConfTourneyWins', 0) - b_conf.iloc[0].get('ConfTourneyWins', 0)
                features['ConfChamp_diff'] = a_conf.iloc[0].get('ConfTourneyChamp', 0) - b_conf.iloc[0].get('ConfTourneyChamp', 0)

        # --- Interaction features ---
        features['Seed_x_Elo'] = features.get('SeedDiff', 0) * features.get('EloDiff', 0)
        features['Seed_x_NetRating'] = features.get('SeedDiff', 0) * features.get('NetRating_diff', 0)

        rows.append(features)

    return pd.DataFrame(rows)

print("\nBuilding Men's matchup features...")
m_train = build_matchup_features(
    m_tourney_compact, m_seeds, m_team_stats, m_elo_df,
    m_massey_features, m_coach_features, m_conf_features, gender='M'
)
print(f"  Men's training data: {m_train.shape}")

print("Building Women's matchup features...")
w_train = build_matchup_features(
    w_tourney_compact, w_seeds, w_team_stats, w_elo_df,
    None, None, None, gender='W'  # Less data available for women
)
print(f"  Women's training data: {w_train.shape}")

# Combine for joint model
m_train['Gender'] = 0
w_train['Gender'] = 1
train_all = pd.concat([m_train, w_train], ignore_index=True)
print(f"\nCombined training data: {train_all.shape}")
print(f"Features: {[c for c in train_all.columns if c not in ['Season', 'TeamA', 'TeamB', 'Target', 'Gender', 'SeedA', 'SeedB', 'EloA', 'EloB']]}")

In [ ]:
# Define feature columns
META_COLS = ['Season', 'TeamA', 'TeamB', 'Target', 'Gender', 'SeedA', 'SeedB', 'EloA', 'EloB']
FEATURE_COLS = [c for c in train_all.columns if c not in META_COLS]
print(f"\n{len(FEATURE_COLS)} features: {FEATURE_COLS}")

# Handle NaN
train_all[FEATURE_COLS] = train_all[FEATURE_COLS].fillna(0)

X = train_all[FEATURE_COLS].values
y = train_all['Target'].values
seasons = train_all['Season'].values
print(f"X shape: {X.shape}, y mean: {y.mean():.3f}")

## 3. Cross-Validation Framework

In [ ]:
def leave_one_season_out_cv(X_df, y, seasons, model_fn, feature_cols=None, val_start=2015):
    """Leave-one-season-out CV. Returns OOF predictions and per-fold scores."""
    if feature_cols is not None:
        X = X_df[feature_cols].values if isinstance(X_df, pd.DataFrame) else X_df
    else:
        X = X_df

    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= val_start))
    results = []
    oof_preds = np.full(len(y), np.nan)

    for val_season in val_seasons:
        train_mask = seasons < val_season
        val_mask = seasons == val_season

        if val_mask.sum() == 0:
            continue

        X_tr, X_val = X[train_mask], X[val_mask]
        y_tr, y_val = y[train_mask], y[val_mask]

        model = model_fn()
        model.fit(X_tr, y_tr)
        preds = model.predict_proba(X_val)[:, 1]
        preds = np.clip(preds, PRED_CLIP_MIN, PRED_CLIP_MAX)

        bs = np.mean((y_val - preds) ** 2)
        ll = log_loss(y_val, preds)
        auc = roc_auc_score(y_val, preds) if len(np.unique(y_val)) > 1 else 0.5

        results.append({'Season': val_season, 'N': val_mask.sum(),
                        'Brier': bs, 'LogLoss': ll, 'AUC': auc})
        oof_preds[val_mask] = preds

    results_df = pd.DataFrame(results)
    valid_mask = ~np.isnan(oof_preds)
    overall_brier = np.mean((y[valid_mask] - oof_preds[valid_mask]) ** 2)
    overall_ll = log_loss(y[valid_mask], oof_preds[valid_mask])
    overall_auc = roc_auc_score(y[valid_mask], oof_preds[valid_mask])

    return {
        'per_fold': results_df,
        'overall': {'Brier': overall_brier, 'LogLoss': overall_ll, 'AUC': overall_auc},
        'oof_preds': oof_preds,
        'valid_mask': valid_mask,
    }

## 4. Traditional ML Models - Phase 4

### 4.1 Baseline: Seed-Only Logistic Regression

In [ ]:
print("=" * 60)
print("MODEL 1: Seed-Only Logistic Regression (Baseline)")
print("=" * 60)

seed_lr_result = leave_one_season_out_cv(
    train_all, y, seasons,
    lambda: LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
    feature_cols=['SeedDiff']
)
print(f"  Overall Brier: {seed_lr_result['overall']['Brier']:.4f}")
print(f"  Overall LogLoss: {seed_lr_result['overall']['LogLoss']:.4f}")
print(f"  Overall AUC: {seed_lr_result['overall']['AUC']:.4f}")
print(seed_lr_result['per_fold'].to_string(index=False))

### 4.2 Full Logistic Regression

In [ ]:
print("\n" + "=" * 60)
print("MODEL 2: Full Logistic Regression")
print("=" * 60)

full_lr_result = leave_one_season_out_cv(
    train_all, y, seasons,
    lambda: LogisticRegression(C=0.5, penalty='l2', max_iter=1000, random_state=SEED),
    feature_cols=FEATURE_COLS
)
print(f"  Overall Brier: {full_lr_result['overall']['Brier']:.4f}")
print(f"  Overall LogLoss: {full_lr_result['overall']['LogLoss']:.4f}")
print(f"  Overall AUC: {full_lr_result['overall']['AUC']:.4f}")

### 4.3 XGBoost

In [ ]:
from xgboost import XGBClassifier

print("\n" + "=" * 60)
print("MODEL 3: XGBoost")
print("=" * 60)

xgb_result = leave_one_season_out_cv(
    train_all, y, seasons,
    lambda: XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        objective='binary:logistic', eval_metric='logloss',
        tree_method='hist', random_state=SEED, verbosity=0
    ),
    feature_cols=FEATURE_COLS
)
print(f"  Overall Brier: {xgb_result['overall']['Brier']:.4f}")
print(f"  Overall LogLoss: {xgb_result['overall']['LogLoss']:.4f}")
print(f"  Overall AUC: {xgb_result['overall']['AUC']:.4f}")

### 4.4 LightGBM

In [ ]:
from lightgbm import LGBMClassifier

print("\n" + "=" * 60)
print("MODEL 4: LightGBM")
print("=" * 60)

lgb_result = leave_one_season_out_cv(
    train_all, y, seasons,
    lambda: LGBMClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        num_leaves=31, subsample=0.8, colsample_bytree=0.8,
        min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
        objective='binary', metric='binary_logloss',
        verbose=-1, random_state=SEED
    ),
    feature_cols=FEATURE_COLS
)
print(f"  Overall Brier: {lgb_result['overall']['Brier']:.4f}")
print(f"  Overall LogLoss: {lgb_result['overall']['LogLoss']:.4f}")
print(f"  Overall AUC: {lgb_result['overall']['AUC']:.4f}")

### 4.5 CatBoost

In [ ]:
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("CatBoost not installed, skipping.")

if HAS_CATBOOST:
    print("\n" + "=" * 60)
    print("MODEL 5: CatBoost")
    print("=" * 60)

    cb_result = leave_one_season_out_cv(
        train_all, y, seasons,
        lambda: CatBoostClassifier(
            iterations=300, learning_rate=0.05, depth=5,
            l2_leaf_reg=3.0, loss_function='Logloss',
            random_seed=SEED, verbose=0
        ),
        feature_cols=FEATURE_COLS
    )
    print(f"  Overall Brier: {cb_result['overall']['Brier']:.4f}")
    print(f"  Overall LogLoss: {cb_result['overall']['LogLoss']:.4f}")
    print(f"  Overall AUC: {cb_result['overall']['AUC']:.4f}")

### 4.6 Elo-Only Model

In [ ]:
print("\n" + "=" * 60)
print("MODEL 6: Elo-Only")
print("=" * 60)

elo_lr_result = leave_one_season_out_cv(
    train_all, y, seasons,
    lambda: LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
    feature_cols=['EloDiff']
)
print(f"  Overall Brier: {elo_lr_result['overall']['Brier']:.4f}")
print(f"  Overall LogLoss: {elo_lr_result['overall']['LogLoss']:.4f}")
print(f"  Overall AUC: {elo_lr_result['overall']['AUC']:.4f}")

### 4.7 Model Comparison

In [ ]:
model_results = {
    'Seed-LR (baseline)': seed_lr_result,
    'Elo-LR': elo_lr_result,
    'Full-LR': full_lr_result,
    'XGBoost': xgb_result,
    'LightGBM': lgb_result,
}
if HAS_CATBOOST:
    model_results['CatBoost'] = cb_result

comparison = pd.DataFrame({
    name: res['overall'] for name, res in model_results.items()
}).T.sort_values('Brier')

print("\n" + "=" * 60)
print("MODEL COMPARISON (sorted by Brier score)")
print("=" * 60)
print(comparison.to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
comparison['Brier'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Brier Score (lower = better)'); axes[0].set_ylabel('Brier')
comparison['LogLoss'].plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Log Loss (lower = better)'); axes[1].set_ylabel('LogLoss')
comparison['AUC'].plot(kind='bar', ax=axes[2], color='green', edgecolor='black')
axes[2].set_title('AUC (higher = better)'); axes[2].set_ylabel('AUC')
plt.suptitle('Model Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.8 Feature Importance (SHAP)

In [ ]:
# Train final XGBoost on all data for SHAP
xgb_final = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    objective='binary:logistic', tree_method='hist', random_state=SEED, verbosity=0
)
xgb_final.fit(train_all[FEATURE_COLS].fillna(0), y)

# Built-in feature importance
importance = pd.Series(xgb_final.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 10))
importance.head(25).plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Top 25 XGBoost Feature Importances', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUT_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Hyperparameter Tuning with Optuna

In [ ]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 7),
            'subsample': trial.suggest_float('subsample', 0.6, 0.95),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma': trial.suggest_float('gamma', 0, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        }

        result = leave_one_season_out_cv(
            train_all, y, seasons,
            lambda: XGBClassifier(**params, objective='binary:logistic',
                                   tree_method='hist', random_state=SEED, verbosity=0),
            feature_cols=FEATURE_COLS, val_start=2018
        )
        return result['overall']['Brier']

    print("Running Optuna XGBoost tuning (50 trials)...")
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=50, timeout=600)

    print(f"\nBest Brier: {study.best_value:.4f}")
    print(f"Best params: {study.best_params}")

    # Retrain with best params
    best_xgb_result = leave_one_season_out_cv(
        train_all, y, seasons,
        lambda: XGBClassifier(**study.best_params, objective='binary:logistic',
                               tree_method='hist', random_state=SEED, verbosity=0),
        feature_cols=FEATURE_COLS
    )
    print(f"Tuned XGBoost Brier: {best_xgb_result['overall']['Brier']:.4f}")
    model_results['XGBoost-Tuned'] = best_xgb_result

except ImportError:
    print("Optuna not available. Using default XGBoost params.")
    best_xgb_result = xgb_result

## 6. Ensemble & Calibration - Phase 6

In [ ]:
# Collect OOF predictions from all models
print("=" * 60)
print("ENSEMBLE CONSTRUCTION")
print("=" * 60)

oof_dict = {}
for name, res in model_results.items():
    mask = res['valid_mask']
    if mask.sum() > 0:
        oof_dict[name] = res['oof_preds'].copy()
        print(f"  {name}: {mask.sum()} OOF predictions")

# Prediction correlation matrix
valid = model_results['XGBoost']['valid_mask']
corr_df = pd.DataFrame({k: v[valid] for k, v in oof_dict.items()})
print("\nModel Prediction Correlations:")
print(corr_df.corr().round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_df.corr(), annot=True, fmt='.3f', cmap='coolwarm', ax=ax)
ax.set_title('Model Prediction Correlation', fontsize=14)
plt.tight_layout()
plt.show()

### 6.1 Optimize Ensemble Weights

In [ ]:
def optimize_weights(oof_dict, y_true, valid_mask):
    """Find optimal ensemble weights minimizing Brier score."""
    names = list(oof_dict.keys())
    preds_matrix = np.column_stack([oof_dict[n][valid_mask] for n in names])
    y_val = y_true[valid_mask]

    def objective(weights):
        w = weights / weights.sum()
        ensemble = np.clip(preds_matrix @ w, PRED_CLIP_MIN, PRED_CLIP_MAX)
        return np.mean((y_val - ensemble) ** 2)

    n = len(names)
    x0 = np.ones(n) / n
    bounds = [(0, 1)] * n
    constraints = {'type': 'eq', 'fun': lambda w: w.sum() - 1.0}

    result = minimize(objective, x0, bounds=bounds, constraints=constraints, method='SLSQP')
    weights = dict(zip(names, result.x))
    return weights, result.fun

opt_weights, opt_brier = optimize_weights(oof_dict, y, valid)
print("Optimized Ensemble Weights:")
for name, w in sorted(opt_weights.items(), key=lambda x: -x[1]):
    print(f"  {name:25s}: {w:.3f}")
print(f"\nOptimized Ensemble Brier: {opt_brier:.4f}")

# Simple average ensemble
simple_preds = np.mean([oof_dict[n][valid] for n in oof_dict], axis=0)
simple_preds = np.clip(simple_preds, PRED_CLIP_MIN, PRED_CLIP_MAX)
simple_brier = np.mean((y[valid] - simple_preds) ** 2)
print(f"Simple Average Brier: {simple_brier:.4f}")

### 6.2 Calibration Analysis

In [ ]:
def plot_calibration(y_true, y_pred, name, ax):
    """Plot reliability diagram."""
    from sklearn.calibration import calibration_curve
    fraction_pos, mean_pred = calibration_curve(y_true, y_pred, n_bins=10)
    ax.plot(mean_pred, fraction_pos, 's-', label=name)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Actual Frequency')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot calibration for each model
for name, preds in oof_dict.items():
    if valid.sum() > 0:
        plot_calibration(y[valid], preds[valid], name, axes[0])
axes[0].legend(fontsize=8)
axes[0].set_title('Calibration Curves (Before)')

# Ensemble calibration
ens_preds = np.zeros_like(y, dtype=float)
for name, w in opt_weights.items():
    ens_preds += w * oof_dict[name]
ens_preds = np.clip(ens_preds[valid], PRED_CLIP_MIN, PRED_CLIP_MAX)
plot_calibration(y[valid], ens_preds, 'Weighted Ensemble', axes[1])
plot_calibration(y[valid], simple_preds, 'Simple Average', axes[1])
axes[1].legend()
axes[1].set_title('Ensemble Calibration')
plt.tight_layout()
plt.savefig(OUT_DIR / 'calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Generate Submission - Phase 7

In [ ]:
def generate_predictions(sub_df, team_stats, seeds_df, elo_df, massey_df,
                          coach_df, conf_df, models_dict, weights, gender_id):
    """Generate predictions for all matchups in submission file."""
    predictions = []

    for _, row in sub_df.iterrows():
        parts = row['ID'].split('_')
        season = int(parts[0])
        team_a, team_b = int(parts[1]), int(parts[2])

        # Determine gender
        is_mens = 1000 <= team_a <= 1999

        # Build features for this matchup (same as training)
        features = {}
        features['SeedDiff'] = 0
        features['EloDiff'] = 0

        # Seeds
        a_seed = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_a)]
        b_seed = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_b)]
        if len(a_seed) > 0 and len(b_seed) > 0:
            features['SeedDiff'] = a_seed.iloc[0]['SeedNum'] - b_seed.iloc[0]['SeedNum']
            features['SeedA'] = a_seed.iloc[0]['SeedNum']
            features['SeedB'] = b_seed.iloc[0]['SeedNum']

        # Elo
        a_elo = elo_df[(elo_df['Season'] == season) & (elo_df['TeamID'] == team_a)]
        b_elo = elo_df[(elo_df['Season'] == season) & (elo_df['TeamID'] == team_b)]
        if len(a_elo) > 0 and len(b_elo) > 0:
            features['EloDiff'] = a_elo.iloc[0]['EloRating'] - b_elo.iloc[0]['EloRating']

        # Team stats
        a_stats = team_stats[(team_stats['Season'] == season) & (team_stats['TeamID'] == team_a)]
        b_stats = team_stats[(team_stats['Season'] == season) & (team_stats['TeamID'] == team_b)]

        stat_cols = ['WinPct', 'PointDiff', 'eFG_pct', 'TO_pct', 'ORB_pct', 'FT_rate',
                     'OffRating', 'DefRating', 'NetRating', 'Pace', 'FG_pct', 'FG3_pct',
                     'FT_pct', 'Ast_TO_ratio', 'Opp_eFG_pct', 'Opp_TO_pct',
                     'Last10_WinPct', 'Last10_PointDiff', 'MarginStd', 'RoadWinPct']

        if len(a_stats) > 0 and len(b_stats) > 0:
            a, b = a_stats.iloc[0], b_stats.iloc[0]
            for col in stat_cols:
                if col in a.index and col in b.index:
                    features[f'{col}_diff'] = a[col] - b[col]

        # Massey
        if massey_df is not None and is_mens:
            a_m = massey_df[(massey_df['Season'] == season) & (massey_df['TeamID'] == team_a)]
            b_m = massey_df[(massey_df['Season'] == season) & (massey_df['TeamID'] == team_b)]
            if len(a_m) > 0 and len(b_m) > 0:
                am, bm = a_m.iloc[0], b_m.iloc[0]
                for sys in ['POM', 'SAG', 'MOR', 'ConsensusRank']:
                    if sys in am.index and sys in bm.index:
                        features[f'{sys}_diff'] = (am[sys] if not pd.isna(am[sys]) else 150) - \
                                                   (bm[sys] if not pd.isna(bm[sys]) else 150)

        # Coach
        if coach_df is not None and is_mens:
            a_c = coach_df[(coach_df['Season'] == season) & (coach_df['TeamID'] == team_a)]
            b_c = coach_df[(coach_df['Season'] == season) & (coach_df['TeamID'] == team_b)]
            if len(a_c) > 0 and len(b_c) > 0:
                features['CoachExp_diff'] = a_c.iloc[0]['CoachTourneyWins'] - b_c.iloc[0]['CoachTourneyWins']

        # Conf tourney
        if conf_df is not None and is_mens:
            a_cf = conf_df[(conf_df['Season'] == season) & (conf_df['TeamID'] == team_a)]
            b_cf = conf_df[(conf_df['Season'] == season) & (conf_df['TeamID'] == team_b)]
            if len(a_cf) > 0 and len(b_cf) > 0:
                features['ConfTourneyWins_diff'] = a_cf.iloc[0].get('ConfTourneyWins', 0) - b_cf.iloc[0].get('ConfTourneyWins', 0)
                features['ConfChamp_diff'] = a_cf.iloc[0].get('ConfTourneyChamp', 0) - b_cf.iloc[0].get('ConfTourneyChamp', 0)

        # Interactions
        features['Seed_x_Elo'] = features.get('SeedDiff', 0) * features.get('EloDiff', 0)
        features['Seed_x_NetRating'] = features.get('SeedDiff', 0) * features.get('NetRating_diff', 0)

        predictions.append(features)

    # Convert to DataFrame and predict
    pred_df = pd.DataFrame(predictions)
    for col in FEATURE_COLS:
        if col not in pred_df.columns:
            pred_df[col] = 0
    pred_df = pred_df[FEATURE_COLS].fillna(0)

    # Get ensemble prediction
    all_preds = {}
    for name, model in models_dict.items():
        all_preds[name] = model.predict_proba(pred_df.values)[:, 1]

    # Weighted ensemble
    final_preds = np.zeros(len(pred_df))
    for name, w in weights.items():
        if name in all_preds:
            final_preds += w * all_preds[name]
        else:
            # Distribute missing weight equally
            pass

    # Normalize if weights don't sum to 1 for available models
    total_w = sum(w for n, w in weights.items() if n in all_preds)
    if total_w > 0:
        final_preds = final_preds / total_w

    final_preds = np.clip(final_preds, PRED_CLIP_MIN, PRED_CLIP_MAX)
    return final_preds

### 7.1 Train Final Models on All Data

In [ ]:
print("Training final models on all data...")

# Retrain all models on full training set
final_models = {}
X_full = train_all[FEATURE_COLS].fillna(0).values

# Logistic Regression
lr = LogisticRegression(C=0.5, penalty='l2', max_iter=1000, random_state=SEED)
lr.fit(X_full, y)
final_models['Full-LR'] = lr

# XGBoost (use tuned params if available)
xgb_params = study.best_params if 'study' in dir() else {
    'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 5,
    'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 3,
    'gamma': 0.1, 'reg_alpha': 0.1, 'reg_lambda': 1.0
}
xgb_f = XGBClassifier(**xgb_params, objective='binary:logistic',
                        tree_method='hist', random_state=SEED, verbosity=0)
xgb_f.fit(X_full, y)
final_models['XGBoost'] = xgb_f
if 'study' in dir():
    final_models['XGBoost-Tuned'] = xgb_f

# LightGBM
lgb_f = LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
                         num_leaves=31, subsample=0.8, colsample_bytree=0.8,
                         min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
                         verbose=-1, random_state=SEED)
lgb_f.fit(X_full, y)
final_models['LightGBM'] = lgb_f

# CatBoost
if HAS_CATBOOST:
    cb_f = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=5,
                                l2_leaf_reg=3.0, random_seed=SEED, verbose=0)
    cb_f.fit(X_full, y)
    final_models['CatBoost'] = cb_f

# Seed-LR and Elo-LR (simple models)
seed_lr_f = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
seed_lr_f.fit(train_all[['SeedDiff']].fillna(0).values, y)

elo_lr_f = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
elo_lr_f.fit(train_all[['EloDiff']].fillna(0).values, y)

print(f"  Trained {len(final_models)} ensemble models")

### 7.2 Generate Stage 1 & Stage 2 Submissions

In [ ]:
# Combine men's and women's data for prediction
all_seeds = pd.concat([m_seeds, w_seeds], ignore_index=True)
all_elo = pd.concat([m_elo_df, w_elo_df], ignore_index=True)
all_team_stats = pd.concat([m_team_stats, w_team_stats], ignore_index=True)

# Use equal weights as fallback if optimization used different model names
pred_weights = {}
for name in final_models:
    if name in opt_weights:
        pred_weights[name] = opt_weights[name]
    else:
        pred_weights[name] = 1.0 / len(final_models)
# Normalize
total = sum(pred_weights.values())
pred_weights = {k: v/total for k, v in pred_weights.items()}

print("Generating Stage 1 predictions...")
s1_preds = generate_predictions(
    sub1, all_team_stats, all_seeds, all_elo,
    m_massey_features, m_coach_features, m_conf_features,
    final_models, pred_weights, gender_id=None
)
sub1['Pred'] = s1_preds
sub1.to_csv(OUT_DIR / 'submission_stage1.csv', index=False)
print(f"  Stage 1: {len(sub1)} predictions, mean={s1_preds.mean():.4f}, std={s1_preds.std():.4f}")

print("Generating Stage 2 predictions...")
s2_preds = generate_predictions(
    sub2, all_team_stats, all_seeds, all_elo,
    m_massey_features, m_coach_features, m_conf_features,
    final_models, pred_weights, gender_id=None
)
sub2['Pred'] = s2_preds
sub2.to_csv(OUT_DIR / 'submission_stage2.csv', index=False)
print(f"  Stage 2: {len(sub2)} predictions, mean={s2_preds.mean():.4f}, std={s2_preds.std():.4f}")

### 7.3 Conservative Submission (Blend with Seed Prior)

In [ ]:
# Historical seed win probability as prior
def seed_prior(team_a, team_b, seeds_df, season):
    a_seed = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_a)]
    b_seed = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_b)]
    if len(a_seed) > 0 and len(b_seed) > 0:
        diff = a_seed.iloc[0]['SeedNum'] - b_seed.iloc[0]['SeedNum']
        return 1.0 / (1.0 + 10.0 ** (diff * 0.15))
    return 0.5

# Build seed-based predictions for Stage 2
seed_preds_s2 = []
for _, row in sub2.iterrows():
    parts = row['ID'].split('_')
    season, ta, tb = int(parts[0]), int(parts[1]), int(parts[2])
    seed_preds_s2.append(seed_prior(ta, tb, all_seeds, season))
seed_preds_s2 = np.array(seed_preds_s2)

# Conservative: 30% seed prior + 70% model
conservative_preds = 0.30 * seed_preds_s2 + 0.70 * s2_preds
conservative_preds = np.clip(conservative_preds, PRED_CLIP_MIN, PRED_CLIP_MAX)

sub2_conservative = sub2.copy()
sub2_conservative['Pred'] = conservative_preds
sub2_conservative.to_csv(OUT_DIR / 'submission_stage2_conservative.csv', index=False)
print(f"Conservative submission: mean={conservative_preds.mean():.4f}, std={conservative_preds.std():.4f}")

### 7.4 Submission Validation

In [ ]:
print("\n" + "=" * 60)
print("SUBMISSION VALIDATION")
print("=" * 60)

for name, sub, preds in [('Stage1', sub1, s1_preds),
                          ('Stage2-Aggressive', sub2, s2_preds),
                          ('Stage2-Conservative', sub2_conservative, conservative_preds)]:
    print(f"\n{name}:")
    print(f"  Rows: {len(sub)}")
    print(f"  Pred range: [{preds.min():.4f}, {preds.max():.4f}]")
    print(f"  Pred mean: {preds.mean():.4f}")
    print(f"  Pred std: {preds.std():.4f}")
    print(f"  Any NaN: {np.isnan(preds).sum()}")
    print(f"  In [0.05, 0.95]: {((preds >= 0.05) & (preds <= 0.95)).all()}")

## 8. Final Summary

In [ ]:
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

print("\nCV Results (Leave-One-Season-Out):")
print(comparison.to_string())

print(f"\nEnsemble Weights: {opt_weights}")
print(f"Ensemble Brier: {opt_brier:.4f}")

print(f"\nSubmissions saved:")
print(f"  {OUT_DIR / 'submission_stage1.csv'}")
print(f"  {OUT_DIR / 'submission_stage2.csv'} (aggressive)")
print(f"  {OUT_DIR / 'submission_stage2_conservative.csv'} (conservative)")
print(f"\nSelect 2 submissions for final scoring:")
print(f"  1. Conservative (safe floor)")
print(f"  2. Aggressive (higher ceiling)")